# Обновление таблиц в PostGIS
## Загружаем данные из PostGIS в Spark (temp view), обрабатываем на Sedona и обновляем таблицы в PostGIS

# Инициализация

In [1]:
import os
import sys

from sedona.spark import SedonaContext

# Явно указываем путь к Python для воркеров
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Настройка Hadoop
os.environ['HADOOP_HOME'] = r'C:\Hadoop\hadoop-3.3.6'

# Пакеты для Spark 3.5.4 со Scala 2.12
additional_packages = [
    "org.apache.sedona:sedona-spark-3.5_2.12:1.8.0",
    "org.datasyslab:geotools-wrapper:1.8.0-33.1"
]

config = SedonaContext.builder() \
    .appName("SedonaApp") \
    .config("spark.jars.packages", ",".join(additional_packages)) \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
    .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
    .master("local[*]") \
    .getOrCreate()

sedona = SedonaContext.create(config)


# Декоратор для замера скорости выполнения запроса

In [2]:
import time
from functools import wraps

def timer_sedona(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнена за {end - start:.4f} секунд")
        return result
    return wrapper

In [3]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226?currentSchema=egip"

# Загрузка данных

In [29]:
# @timer_sedona
def download_data(session, query):
    result_df = session.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({query}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
    result_df.createOrReplaceTempView("spatial_table")

# Выполнение запроса

In [5]:
@timer_sedona
def execute_query(session, download_query, query):
    # new_df = loaded_df.selectExpr(query)
    download_data(session, download_query)
    result_df = session.sql(query)
    result_df.show(10)
    return result_df

# Универсальная функция обновления таблицы 

In [78]:
import psycopg2
from typing import Dict
@timer_sedona
def update_table_by_join_universal(
    df_updates, 
    target_database='gisdb_8411_250226', 
    target_table="public.features_plain_test",
    join_keys=["id", "layer_id"],
    update_mappings: Dict[str, str] = None,
    batch_size: int = 10000
):
    """
    Универсальная функция для обновления таблицы по JOIN
    
    Args:
        df_updates: DataFrame с данными для обновления
        target_database: Имя базы данных
        target_table: Имя целевой таблицы
        join_keys: Список ключей для JOIN
        update_mappings: Словарь соответствия {source_column: target_column}
                       Например: {"area": "polygon_area", "perimeter": "polygon_perimeter"}
        batch_size: Размер батча
    
    Example:
        update_mappings = {
            "area": "polygon_area",
            "perimeter": "polygon_perimeter", 
            "centroid": "geometry_centroid",
            "area_3857": "area_3857"
        }
    """
    
    if not update_mappings:
        raise ValueError("update_mappings cannot be empty")
    
    # Подготавливаем данные
    source_columns = list(update_mappings.keys())
    all_columns = join_keys + source_columns
    rows = df_updates.select(*all_columns).collect()
    
    if not rows:
        print("No data to update")
        return 0
    
    # Подключаемся к БД
    conn = psycopg2.connect(
        host=credentials.get('host'),
        database=target_database,
        user=credentials.get("user"),
        password=credentials.get("password"),
        port=5432
    )
    
    def format_value(value):
        """Форматирует значение для SQL с правильным экранированием"""
        if value is None:
            return 'NULL'
        elif isinstance(value, str):
            # Экранируем одинарные кавычки и оборачиваем строку в кавычки
            escaped = value.replace("'", "''")
            return f"'{escaped}'"
        elif isinstance(value, (int, float)):
            return str(value)
        else:
            # Для других типов преобразуем в строку и экранируем
            escaped = str(value).replace("'", "''")
            return f"'{escaped}'"
    
    try:
        cur = conn.cursor()
        total_updated = 0
        
        # Обрабатываем батчами
        for i in range(0, len(rows), batch_size):
            batch = rows[i:i+batch_size]
            
            # Формируем VALUES для батча
            values_parts = []
            for row in batch:
                # Берем значения для join ключей
                join_values = [row[key] for key in join_keys]
                # Берем значения для обновляемых полей
                update_values = [row[col] for col in source_columns]
                # Формируем строку VALUES с правильным форматированием
                all_values = join_values + update_values
                formatted_values = [format_value(v) for v in all_values]
                values_parts.append(f"({', '.join(formatted_values)})")
            
            # Формируем список колонок для CTE
            cte_columns = join_keys + source_columns
            cte_columns_str = ', '.join(cte_columns)
            
            # Формируем SET часть
            set_parts = []
            for source_col, target_col in update_mappings.items():
                set_parts.append(f"{target_col} = u.{source_col}")
            set_clause = ', '.join(set_parts)
            
            # Формируем WHERE часть для проверки изменений
            where_parts = []
            for source_col, target_col in update_mappings.items():
                where_parts.append(f"f.{target_col} IS DISTINCT FROM u.{source_col}")
            where_changes = ' OR '.join(where_parts) if where_parts else 'TRUE'
            
            # Формируем условие JOIN
            join_conditions = ' AND '.join([f"f.{key} = u.{key}" for key in join_keys])
            
            # Формируем запрос
            update_query = f"""
                WITH updates ({cte_columns_str}) AS (
                    VALUES 
                    {', '.join(values_parts)}
                )
                UPDATE {target_table} f
                SET {set_clause}
                FROM updates u
                WHERE {join_conditions}
                AND ({where_changes})
            """
            
            # Выполняем один запрос для батча
            cur.execute(update_query)
            batch_updated = cur.rowcount
            total_updated += batch_updated
            conn.commit()
            
            print(f"Batch {i//batch_size + 1}: updated {batch_updated} rows")
        
        print(f"✅ Total updated: {total_updated} rows in {target_table}")
        return total_updated
        
    except Exception as e:
        conn.rollback()
        print(f"Error: {e}")
        raise
    finally:
        cur.close()
        conn.close()

# Обновление площади

In [26]:
# запрос
# sql_area = """SELECT id, layer_id, ST_Area(ST_Transform(ST_SetSRID(geometry, 4326), 'EPSG:3857')) as area FROM public.features_plain LIMIT """
sql_download_data = """ SELECT id, layer_id, geometry_3857 FROM public.features_plain_test LIMIT 100000"""
sql_area = """SELECT id, layer_id, ST_Area(ST_SetSRID(ST_GeomFromWKB(geometry_3857), 3857)) as polygon_area FROM spatial_table"""
area_df = execute_query(sedona, sql_download_data, sql_area)

+-------+--------+------------+
|     id|layer_id|polygon_area|
+-------+--------+------------+
|3586459|     168|         0.0|
|3586660|     168|         0.0|
|3586661|     168|         0.0|
|3586662|     168|         0.0|
|3586663|     168|         0.0|
|3586733|     168|         0.0|
|3586664|     168|         0.0|
|3586665|     168|         0.0|
|3586666|     168|         0.0|
|3586668|     168|         0.0|
+-------+--------+------------+
only showing top 10 rows

Функция execute_query выполнена за 16.2029 секунд


In [27]:
# Обновление
update_table_by_join_universal(
    df_updates=area_df,
    target_database = 'gisdb_8411_250226',
    target_table="public.features_plain_test",
    join_keys=["id", "layer_id"],
    update_mappings=dict({"polygon_area":"polygon_area"})
)

Batch 1: updated 10000 rows
Batch 2: updated 10000 rows
Batch 3: updated 10000 rows
Batch 4: updated 10000 rows
Batch 5: updated 10000 rows
Batch 6: updated 10000 rows
Batch 7: updated 10000 rows
Batch 8: updated 10000 rows
Batch 9: updated 10000 rows
Batch 10: updated 10000 rows
✅ Total updated: 100000 rows in public.features_plain_test
Функция update_table_by_join_universal выполнена за 26.3295 секунд


100000

# Обновление периметра

In [42]:
sql_download_data = """ SELECT id, layer_id, geometry_3857 FROM public.features_plain_test LIMIT 100000"""
sql_area = """SELECT id, layer_id, ST_Perimeter(ST_GeomFromWKB(geometry_3857)) as polygon_perimeter FROM spatial_table"""
area_df = execute_query(sedona, sql_download_data, sql_area)

+-------+--------+------------------+
|     id|layer_id| polygon_perimeter|
+-------+--------+------------------+
|3684953|     170| 765.3256946569825|
|3409984|     164|               0.0|
|3573026|     168|               0.0|
|3664395|     170| 733.0413140261076|
|3586883|     168|               0.0|
|3569734|     168|               0.0|
|3516122|     165| 415.0707581091008|
|3692524|     170|1018.5595635689895|
|3588447|     168|               0.0|
|3683453|     170| 469.9736821406416|
+-------+--------+------------------+
only showing top 10 rows

Функция execute_query выполнена за 16.3305 секунд


In [44]:
# Обновление
update_table_by_join_universal(
    df_updates=area_df,
    target_database = 'gisdb_8411_250226',
    target_table="public.features_plain_test",
    join_keys=["id", "layer_id"],
    update_mappings=dict({"polygon_perimeter":"polygon_perimeter"})
)

Batch 1: updated 10000 rows
Batch 2: updated 10000 rows
Batch 3: updated 10000 rows
Batch 4: updated 10000 rows
Batch 5: updated 10000 rows
Batch 6: updated 10000 rows
Batch 7: updated 10000 rows
Batch 8: updated 10000 rows
Batch 9: updated 10000 rows
Batch 10: updated 10000 rows
✅ Total updated: 100000 rows in public.features_plain_test
Функция update_table_by_join_universal выполнена за 31.1420 секунд


100000

# Обновление координат центроида полигона

In [76]:
sql_download_data = """ SELECT id, layer_id, geometry_3857 FROM public.features_plain_test LIMIT 100000"""
sql_area = """
WITH calc_centroid AS (
    SELECT 
        id,
        layer_id,
        ST_Centroid(ST_GeomFromWKB(geometry_3857)) AS centroid 
    FROM spatial_table
)
SELECT 
    id,
    layer_id,
    ST_X(centroid) AS X_centroid,
    ST_Y(centroid) AS Y_centroid
FROM calc_centroid
"""
area_df = execute_query(sedona, sql_download_data, sql_area)

+-------+--------+------------------+------------------+
|     id|layer_id|        X_centroid|        Y_centroid|
+-------+--------+------------------+------------------+
|3511081|     165|4192237.1858289544| 7483193.378121675|
|3588371|     168|4165632.6983991484|   7504723.6831661|
|3667645|     170|4145490.4486693637| 7480607.279955732|
|3580179|     168| 4181444.391410608| 7489406.528917435|
|3580186|     168| 4172135.386378821| 7513169.858411693|
|3588391|     168| 4193566.117148218| 7493954.648333736|
|3570663|     168|4167059.7096307976| 7529013.530538592|
|3738864|     170|4165392.5391808474|7494404.6189276595|
|3587093|     168| 4217183.333035616| 7502385.784337131|
|3579282|     168| 4177812.401887912| 7494060.298849967|
+-------+--------+------------------+------------------+
only showing top 10 rows

Функция execute_query выполнена за 12.9027 секунд


In [77]:
# Обновление
update_table_by_join_universal(
    df_updates=area_df,
    target_database = 'gisdb_8411_250226',
    target_table="public.features_plain_test",
    join_keys=["id", "layer_id"],
    update_mappings=dict({"X_centroid":"x", "Y_centroid":"y"})
)

Batch 1: updated 10000 rows
Batch 2: updated 10000 rows
Batch 3: updated 10000 rows
Batch 4: updated 10000 rows
Batch 5: updated 10000 rows
Batch 6: updated 10000 rows
Batch 7: updated 10000 rows
Batch 8: updated 10000 rows
Batch 9: updated 10000 rows
Batch 10: updated 10000 rows
✅ Total updated: 100000 rows in public.features_plain_test


100000

# Обновление центроидов

In [87]:
sql_download_data = """ SELECT id, layer_id, geometry_4326, geometry_3857  FROM public.features_plain_test LIMIT 100000"""
sql_area = """
        SELECT 
            id, layer_id,
            ST_AsText(ST_Centroid(ST_GeomFromWKB(geometry_4326))) as centroid_4326, 
            ST_AsText(ST_Centroid(ST_GeomFromWKB(geometry_3857))) as centroid_3857
        FROM spatial_table"""
area_df = execute_query(sedona, sql_download_data, sql_area)

+-------+--------+--------------------+--------------------+
|     id|layer_id|       centroid_4326|       centroid_3857|
+-------+--------+--------------------+--------------------+
|3510036|     165|POINT (37.7701566...|POINT (4204554.60...|
|3686291|     170|POINT (37.5457984...|POINT (4179579.16...|
|3665476|     170|POINT (37.8052058...|POINT (4208456.26...|
|3424113|     164|POINT (37.6255530...|POINT (4188457.41...|
|3577213|     168|POINT (37.8785180...|POINT (4216617.34...|
|3742230|     171|POINT (37.6064625...|POINT (4186332.26...|
|3698629|     170|POINT (37.6948270...|POINT (4196168.95...|
|3708661|     170|POINT (37.6394687...|POINT (4190006.49...|
|3692169|     170|POINT (37.3981541...|POINT (4163143.47...|
|3687770|     170|POINT (37.2937353...|POINT (4151519.62...|
+-------+--------+--------------------+--------------------+
only showing top 10 rows

Функция execute_query выполнена за 25.5998 секунд


In [88]:
# Обновление
update_table_by_join_universal(
    df_updates=area_df,
    target_database = 'gisdb_8411_250226',
    target_table="public.features_plain_test",
    join_keys=["id", "layer_id"],
    update_mappings=dict({"centroid_4326":"centroid_4326", "centroid_3857":"centroid_3857"})
)

Batch 1: updated 10000 rows
Batch 2: updated 10000 rows
Batch 3: updated 10000 rows
Batch 4: updated 10000 rows
Batch 5: updated 10000 rows
Batch 6: updated 10000 rows
Batch 7: updated 10000 rows
Batch 8: updated 10000 rows
Batch 9: updated 10000 rows
Batch 10: updated 10000 rows
✅ Total updated: 100000 rows in public.features_plain_test
Функция update_table_by_join_universal выполнена за 47.9560 секунд


100000

# Центроид полигона в WKT формате с переставленными координатами

In [22]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """
    with flipped_cenroid_cord as (
        select id, layer_id, ST_FlipCoordinates(ST_Centroid(ST_GeomFromWKB(geometry))) as centroid FROM spatial_table )
        select id, layer_id, ST_AsText(centroid) as flipped_centroid FROM flipped_cenroid_cord"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_centroid|
+-------+--------+--------------------+
|3630116|     169|POINT (55.7952289...|
+-------+--------+--------------------+

Функция execute_query выполнена за 1.2814 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_centroid|
+-------+--------+--------------------+
|3630116|     169|POINT (55.7952289...|
|3630117|     169|POINT (55.8030831...|
|3630118|     169|POINT (55.8562968...|
|3630119|     169|POINT (55.8052069...|
|3630120|     169|POINT (55.7411419...|
|3630121|     169|POINT (55.7235098...|
|3630122|     169|POINT (55.6765118...|
|3630123|     169|POINT (55.7328325...|
|3630124|     169|POINT (55.7197321...|
|3630125|     169|POINT (55.7099902...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5223 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_centroid|
+-------+--------+

# Геометрия полигона с переставленными координатами в WKT формате

In [23]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """ SELECT id, layer_id, ST_FlipCoordinates(ST_GeomFromWKB(geometry)) as flipped_geoemtry FROM spatial_table """
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+--------------------+
|3549234|     166|POLYGON ((55.8179...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5707 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+--------------------+
|3549234|     166|POLYGON ((55.8179...|
|3665514|     170|MULTIPOLYGON (((5...|
|3665515|     170|MULTIPOLYGON (((5...|
|3665516|     170|MULTIPOLYGON (((5...|
|3665517|     170|MULTIPOLYGON (((5...|
|3666074|     170|MULTIPOLYGON (((5...|
|3549235|     166|POLYGON ((55.7368...|
|3665522|     170|MULTIPOLYGON (((5...|
|3665523|     170|MULTIPOLYGON (((5...|
|3665524|     170|MULTIPOLYGON (((5...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5702 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+

# Ближайшие объекты. 
## Находим ближайшие объекты к данному с помощью KNN

In [8]:
df = sedona.read \
        .format("jdbc") \
        .option("url", postgresql_url) \
        .option("user", credentials.get('user')) \
        .option("password", credentials.get('password')) \
        .option("dbtable", "public.features_plain") \
        .option("driver", "org.postgresql.Driver") \
        .load()
    
# 2. Регистрируем как временную таблицу
df.createOrReplaceTempView("features_plain")

In [27]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """

sql_area = """
WITH source_geometry AS (
    SELECT ST_GeomFromWKB(geometry) as geometry
    FROM spatial_table
    LIMIT 1
)
SELECT 
    st.id,
    st.layer_id,
    ST_Distance(so.geometry, ST_GeomFromWKB(st.geometry)) as distance
FROM source_geometry so
CROSS JOIN spatial_table st
WHERE ST_Distance(so.geometry, ST_GeomFromWKB(st.geometry)) IS NOT NULL
ORDER BY distance
LIMIT 50
"""
execute_query_all(sedona, sql_download_data, sql_area)


1 запись: 
+-------+--------+--------+
|     id|layer_id|distance|
+-------+--------+--------+
|3510305|     165|     0.0|
+-------+--------+--------+

Функция execute_query выполнена за 0.9507 секунд

 10 записей: 
+-------+--------+-------------------+
|     id|layer_id|           distance|
+-------+--------+-------------------+
|3510305|     165|                0.0|
|3510306|     165|0.10486160776992842|
|3510307|     165|0.10486269364148111|
|3510311|     165|0.11863125629348435|
|3510310|     165|0.12372830248549214|
|3738833|     170|0.12785776251783132|
|3663561|     170|0.16047652972620846|
|3510324|     165|0.16503490930571207|
|3510309|     165| 0.4898143888851502|
|3510308|     165|0.48981658663742417|
+-------+--------+-------------------+

Функция execute_query выполнена за 0.9943 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|            distance|
+-------+--------+--------------------+
|3664650|     170|0.009819354542663737|
|3510569|   

# Упрощение геометрии объекта: ST_SimplifyPreserveTopology

In [29]:
sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """
sql_area = """ SELECT id, layer_id, ST_SimplifyPreserveTopology(ST_GeomFromWKB(geometry), 10) as simplified_geoemtry FROM spatial_table LIMIT """
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3379696|     163|POINT (37.8178420...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5232 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3379696|     163|POINT (37.8178420...|
|3414688|     164|POINT (37.8155465...|
|3414689|     164|POINT (37.4855190...|
|3414690|     164|POINT (37.714637 ...|
|3414691|     164|POINT (37.7143765...|
|3414692|     164|POINT (37.5124900...|
|3414693|     164|POINT (37.5906375...|
|3414694|     164|POINT (37.6882068...|
|3414695|     164|POINT (37.733842 ...|
|3414696|     164|POINT (37.7148106...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.4705 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+

# Определить административную принадлежность объекта к округу и району
## Забираем актуальные границы оркгуов и районов из базы

In [30]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/mkgh_monitorings"

In [31]:
sql_get_regions = """select * from nsi.nsi_moscow_regions"""
df_regions = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_regions}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_regions.show(10)

+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|region_id|           full_name|                name|short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
|  1| 11001200|Троицкий и Новомо...|Троицки

In [32]:
sql_get_districts = """select * from nsi.nsi_moscow_districts"""
df_districts = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_districts}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_districts.show(10)
df_districts.createOrReplaceTempView("districts_table")

+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|district_id|           full_name|          name|        short_name|region_id|   region_name|region_short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------

In [33]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226?currentSchema=egip"

sql_download_data = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT """

sql_query = """
WITH source_objects AS (
    SELECT 
        id,
        layer_id,
        ST_Transform(ST_MakeValid(ST_SetSRID(ST_GeomFromWKB(geometry), 4326)), 'EPSG:3857') AS geometry_3857 
    FROM spatial_table
),
districts_transformed as (
    select  district_id, short_name, region_id, region_short_name,  ST_SetSRID(ST_GeomFromWKB(geometry_3857), 3857) as geometry_3857
    from districts_table
    ),
intersections_with_districts AS (
    SELECT 
        so.id,
        so.layer_id,
        md.district_id,
        md.short_name AS short_district_name,
        md.region_id,
        md.region_short_name AS region_short_name,
        ST_Area(ST_Intersection(so.geometry_3857, md.geometry_3857)) AS area_intersection
    FROM districts_transformed AS md
    CROSS JOIN source_objects AS so
    WHERE ST_Intersects(so.geometry_3857, md.geometry_3857)
),
ranked_intersections AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY id, layer_id ORDER BY area_intersection DESC) AS rn
    FROM intersections_with_districts
)
SELECT 
    id,
    layer_id,
    district_id,
    short_district_name,
    region_id,
    region_short_name,
    area_intersection
FROM ranked_intersections
WHERE rn = 1
"""
execute_query_all(sedona, sql_download_data, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3622300|     169|POINT (36.9414665...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5223 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3622300|     169|POINT (36.9414665...|
|3622301|     169|POINT (37.1872578...|
|3622302|     169|POINT (37.5240918...|
|3622303|     169|POINT (37.524309 ...|
|3622304|     169|POINT (37.3497864...|
|3622305|     169|POINT (37.7191694...|
|3622306|     169|POINT (37.5988063...|
|3622307|     169|POINT (37.4789366...|
|3622308|     169|POINT (36.9365724...|
|3622309|     169|POINT (37.535219 ...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5803 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+